In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pylab as plt
import datetime
import scipy.stats
#import plotly.offline as pyo
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.colors as cl
import matplotlib.dates as mdates
import matplotlib
import statsmodels.api as sm
from statsmodels.regression.rolling import RollingOLS
import seaborn as sns
import netCDF4 as nc
import xarray as xr
#import cartopy.crs as ccrs
#from cartopy import feature as cfa
import os
import plotly.io as pio
pio.renderers.default = 'notebook'

In [ ]:
def decimal_year_to_datetime(decimal_year):
    year = decimal_year.astype(int)
    decimal_part = decimal_year - year
    start_of_year = pd.to_datetime([f'{y}-01-01' for y in year])
    seconds_in_year = (pd.to_datetime([f'{y+1}-01-01' for y in year]) - start_of_year).total_seconds()
    seconds = decimal_part * seconds_in_year
    return start_of_year + pd.to_timedelta(seconds, unit='s')

def datetime_to_decimal_year(datetime_):
    year = datetime_.year
    days = (datetime_.date() - datetime.date(year, 1,1)).days
    res = year + days/365.25
    return res


def FFT_res(x, si):
    n = len(x)
    x_FFT_shifted = np.fft.fftshift(np.fft.fft(x))
    x_FFT_freq_shifted = np.fft.fftshift(np.fft.fftfreq(n, d=si))
    return x_FFT_freq_shifted, x_FFT_shifted


def fft_noramlize(fft_res):
    n = len(fft_res)
    norm_fft = np.abs(fft_res)/(np.max(np.abs(fft_res)))
    return norm_fft

def line_fit(X,Y):
    P = np.polyfit(X,Y,1)
    dx = np.max(X) - np.min(X)
    X_new = np.array([np.min(X)-dx*0.05, np.max(X)+dx*0.05])
    Y_new = np.polyval(P, X_new)
    return X_new, Y_new, P

In [ ]:
%run 20260223_read_data.py
df_sorted

In [ ]:
df_sorted['age_final_month'] = df_sorted['age_dt'].dt.month
df_updated_monthly = df_sorted[['age_final_month', 'd18o_pdb', 'd18o', 'd13c']].groupby('age_final_month').mean()
df_updated_monthly

In [ ]:
morpho_data = pd.read_csv('../../2026-01-08_Luc-Morphology/SeaLevelFromMorphology.csv', header=None)
cols = ['date', 'SPA_luc', 'SPLA_luc','SPA_fred', 'SPLA_fred']
morpho_data.columns = cols
morpho_data

morpho_raw = pd.read_excel('20260306_agemodel/Morphology_Data_ageupdated.xlsx')
#scale_factor_y = 10./(57.69-75.96)
scale_factor_y = (-10.--90.)/(3.03-149.03)
scale_factor_x = -scale_factor_y
offset_y =  -60-scale_factor_y*(94.27)
morpho_raw.loc[:,'x_true'] = morpho_raw.loc[:,'x']*scale_factor_x
morpho_raw.loc[:,'y_true'] = morpho_raw.loc[:,'y']*scale_factor_y + offset_y
morpho_memo = morpho_raw.loc[~morpho_raw['x'].isna(),:].sort_values("age_absolute")
morpho_memo

In [ ]:
climate = pd.read_csv('../../2026-01-15_Climate-Model/analysis/climate_df.csv')
climate['datetime'] = pd.to_datetime(climate['datetime'])
climate['datetime_month'] = climate['datetime'].dt.month
climate_monthly = climate[['SST','SSS','age_absolute', 'synthetic_d18o', 'synthetic_d18o_2','datetime_month']].groupby('datetime_month').mean()
climate_monthly

In [ ]:
tide = pd.read_csv('../../2025-10-10_Fred-TideGauge/memo_edited.csv')
tide['age_dt'] = decimal_year_to_datetime(tide['Date'])#.dt.date
tide = tide.set_index('age_dt')
tide

In [ ]:
enso_phase = pd.read_csv('../../ENSO_Phase_Classification.csv')
enso_phase.columns=['Year','Phase']
enso_phase['Date'] = decimal_year_to_datetime(enso_phase['Year'])
#enso_phase.set_index('Date', inplace=True)
enso_phase['Phase'] = enso_phase['Phase'].str.replace('Warm Phase', 'W').str.replace('Cold Phase', 'C').str.replace('Neutral Phase','N')


enso = pd.read_csv("../../ENSO_Time_Series.csv", parse_dates=[0])
#enso = enso.set_index("Date")
enso['day_of_year'] = enso['Date'].dt.dayofyear
enso['is_leap'] = enso['Date'].dt.is_leap_year
enso['total_days'] = np.where(enso['is_leap'], 366, 365)
enso['decimal_year'] = enso['Date'].dt.year + (enso['day_of_year'] - 1) / enso['total_days']


margin = 0.5
enso['phase'] = 'N'
enso.loc[enso['ENSO']>=margin,'phase'] = "C"
enso.loc[enso['ENSO']<=-margin,'phase'] = "W"
enso['phase_group'] = (enso['phase'] != enso['phase'].shift()).cumsum()
blocks = enso.groupby('phase_group').agg(
    start=('Date', 'first'),
    end=('Date', 'last'),
    value=('phase', 'first')
).reset_index(drop=True)

ENSO_match = np.array([[1982.2, 1984.1],
                       [1986.75, 1988.3],
                       [1992.1, 1992.96],
                       [1997.2, 1998.8]])

enso_fft_freq, enso_fft = FFT_res(enso['ENSO'], 30)


enso


In [ ]:
#print(df_updated_monthly['d18o_pdb'])
#print(climate_monthly['synthetic_d18o'])
print(climate_monthly['synthetic_d18o']/climate_monthly['synthetic_d18o'].min())
print(df_updated_monthly['d18o_pdb']/df_updated_monthly['d18o_pdb'].min())

In [ ]:
fig, ax = plt.subplots(1, figsize=(6,4))


ax.plot(climate_monthly['synthetic_d18o'], color='blue',label='climate')
ax.plot(df_updated_monthly['d18o_pdb']*2+10.2, '--', color='blue', label='d18o obs')
ax.legend(loc='lower right')

ax.set_xlabel('Month of Year')
ax.set_ylabel(r'$\delta^{18}$O')

ax2 = ax.twinx()
#ax2.plot(climate_monthly['d13c'], color='green')
ax2.plot(df_updated_monthly['d13c'],  '--',color='green',label='d13c obs')
#ax2.plot(df_updated_monthly['d18o_pdb']/df_updated_monthly['d18o_pdb'].max(), '--', color='blue', label='model')
ax2.set_ylabel(r'$\delta^{13}$C')
ax2.legend(loc='upper left')

fig.savefig('monthly_climate_model.pdf')

In [ ]:
climate_monthly['synthetic_d18o'].values

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5), subplot_kw={'projection': 'polar'})

ax = axes[0]

months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
          'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

theta = np.linspace(0, 2*np.pi, 12, endpoint=False)
width = 2*np.pi/12 * 0.9  # 90% of bin width (small gap)

rmin, rmax = -0.21 , 0.3
vals = climate_monthly['synthetic_d18o'].values

# Make bars start at rmin, and only show the variation within [rmin, rmax]
heights = np.clip(vals, rmin, rmax) - rmin

ax.bar(theta, heights, width=width, bottom=rmin,
       color='#1A388A', alpha=0.7, edgecolor='black', linewidth=1.5)

ax.set_rlim(rmin, rmax)

ax.set_rticks([])
ax.set_xticks(theta)
ax.set_xticklabels(months)

ax.set_theta_zero_location('N')
ax.set_theta_direction(-1)

ax.set_title(r'climate based $\delta^{18}$O')

ax = axes[1]

months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
          'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

theta = np.linspace(0, 2*np.pi, 12, endpoint=False)
width = 2*np.pi/12 * 0.9  # 90% of bin width (small gap)

rmin, rmax = -5.2, -4.95
vals = df_updated_monthly['d18o_pdb'].values

# Make bars start at rmin, and only show the variation within [rmin, rmax]
heights = np.clip(vals, rmin, rmax) - rmin

ax.bar(theta, heights, width=width, bottom=rmin,
       color='steelblue', alpha=0.7, edgecolor='black', linewidth=1.5)

ax.set_rlim(rmin, rmax)

ax.set_rticks([])
ax.set_xticks(theta)
ax.set_xticklabels(months)

ax.set_theta_zero_location('N')
ax.set_theta_direction(-1)

ax.set_title(r'observed $\delta^{18}$O')


ax = axes[2]


rmin, rmax = -1.45, -1.18
vals = df_updated_monthly['d13c'].values

# Make bars start at rmin, and only show the variation within [rmin, rmax]
heights = np.clip(vals, rmin, rmax) - rmin

ax.bar(theta, heights, width=width, bottom=rmin,
       color='green', alpha=0.7, edgecolor='black', linewidth=1.5)

ax.set_rlim(rmin, rmax)

ax.set_rticks([])
ax.set_xticks(theta)
ax.set_xticklabels(months)

ax.set_theta_zero_location('N')
ax.set_theta_direction(-1)

ax.set_title(r'observed $\delta^{13}$C')

plt.suptitle('Matching Peak Age Model', y= 1.05 , fontsize=16)


#fig.savefig('claude_age_model_d13c_d18o_monthly_means.pdf',bbox_inches='tight' )

In [ ]:

fig = make_subplots(specs=[[{"secondary_y": True}]])

#fig.add_trace(go.Scatter(x=df_sorted['age_dt'], y=df_sorted['d18o']-df_sorted['d18o'].mean(),  mode="markers", marker_color="#3492eb",  name='d18O', text=df_sorted['Name']),secondary_y=False)
#fig.add_trace(go.Scatter(x=df_resampled.index.values, y=df_resampled['d18o_ma']-df_resampled['d18o_ma'].mean(), mode="lines", line_color='#134b80', name='average'), secondary_y=False)
#fig.add_trace(go.Scatter(x=df_resampled.index.values, y=df_resampled['d18o_ma2']-df_resampled['d18o_ma2'].mean(), mode="lines", line_color='navy', name='average'), secondary_y=False)

#fig.add_trace(go.Scatter(x=df_sorted['age_dt'], y=df_sorted['d18o_no_tectonic'] ,  mode="markers", marker_color="#3492eb",  name='d18O', text=df_sorted['Name']),secondary_y=False)
#fig.add_trace(go.Scatter(x=df_sorted['age_dt'], y=df_sorted['d18o_notec_ma'] , mode="lines", line_color='#134b80', name='average'), secondary_y=False)

fig.add_trace(go.Scatter(x=df_sorted['age_final'], y=df_sorted['d18o'] , mode="markers", line_color='#3492eb',text=df_sorted['Name'], name='raw'), secondary_y=False)
fig.add_trace(go.Scatter(x=df_sorted['age_ws'], y=df_sorted['d18o_ma'] , mode="lines", line_color='#134b80', name='average'), secondary_y=False)



fig.add_trace(go.Scatter(x=climate['age_absolute'], y= climate['synthetic_d18o'] , mode="lines", line_color='#F54927', name='synthetic d18o', line_width=1), secondary_y=True)



fig.update_layout(
    yaxis2=dict(title="Secondary Y-axis Title", overlaying="y", side="right", gridcolor='lightgray', showgrid=False)) # Grid for secondary Y-axis)
#fig.update_layout(
#    xaxis2=dict(title="Secondary X-axis Title", overlaying="x", side="top", gridcolor='lightgray', showgrid=False)) # Grid for secondary Y-axis)

fig.update_layout(width=1900, height=800, title_text="d18O observed and d18O synthetic (climate)", template='simple_white')
# Set x-axis title
fig.update_xaxes(title_text="Year")

# Set y-axes titles
fig.update_yaxes(title_text= r'$\delta^{18}$ O Observerd', secondary_y=False)
fig.update_yaxes(title_text= r'$\delta^{18}$ O Synthetic', secondary_y=True)

fig.show()
fig.write_html('d18o_syntetic-d18O-AGEMODELTEST.html')


In [ ]:

fig = make_subplots(specs=[[{"secondary_y": True}]])


fig.add_trace(go.Scatter(x=df_sorted['age_final'], y=df_sorted['d13c'] , mode="markers", line_color='#27F535',text=df_sorted['Name'], name='raw'), secondary_y=False)
fig.add_trace(go.Scatter(x=df_sorted['age_ws'], y=df_sorted['d13c_ma'] , mode="lines", line_color='#05700C', name='3 month average'), secondary_y=False)
fig.add_trace(go.Scatter(x=df_sorted['age_ws2'], y=df_sorted['d13c_ma2'] , mode="lines", line_color='#011803', name='3 year average'), secondary_y=False)

#fig.add_trace(go.Scatter(x=climate['age_absolute'], y= -climate['synthetic_d18o'] , mode="lines", line_color='#F54927', name='synthetic d18o', line_width=1), secondary_y=False)


#fig.add_trace(go.Scatter(x=climate['age_absolute'], y= -climate['synthetic_d18o'] , mode="lines", line_color='#F54927', name='synthetic d18o', line_width=1), secondary_y=True)
fig.add_trace(go.Scatter(x=tide['Date'], y= tide['Sea Level'] , mode="lines", line_color='#27BBF5', name='Sea Level', line_width=2), secondary_y=True)



fig.update_layout(
    yaxis2=dict(title="Secondary Y-axis Title", overlaying="y", side="right", gridcolor='lightgray', showgrid=False)) # Grid for secondary Y-axis)
#fig.update_layout(
#    xaxis2=dict(title="Secondary X-axis Title", overlaying="x", side="top", gridcolor='lightgray', showgrid=False)) # Grid for secondary Y-axis)

fig.update_layout(width=1700, height=800, title_text="d18O observed and d18O synthetic (climate)", template='simple_white')
# Set x-axis title
fig.update_xaxes(title_text="Year")

# Set y-axes titles
fig.update_yaxes(title_text= r'$\delta^{13}$ O Observerd', secondary_y=False)
fig.update_yaxes(title_text= 'Sea Level [mm]', secondary_y=True)

fig.show()
#fig.write_html('d18o_syntetic-d18O.html')


In [ ]:
df_resampled

In [ ]:

fig = make_subplots(specs=[[{"secondary_y": True}]])


fig.add_trace(go.Scatter(x=df_sorted['age_final'], y=df_sorted['d13c'] , mode="markers", line_color='#27F535',text=df_sorted['Name'], name='raw'), secondary_y=False)
fig.add_trace(go.Scatter(x=df_sorted['age_final'], y=df_sorted['d13c'] , mode="lines", line_color='#27F535',text=df_sorted['Name'], name='raw'), secondary_y=False)

fig.add_trace(go.Scatter(x=df_sorted['age_ws'], y=df_sorted['d13c_ma'] , mode="lines", line_color='#05700C', name='3 month average'), secondary_y=False)
fig.add_trace(go.Scatter(x=df_sorted['age_ws2'], y=df_sorted['d13c_ma2'] , mode="lines", line_color='#011803', name='3 year average'), secondary_y=False)

fig.add_trace(go.Scatter(x=df_resampled['age_resampled'], y=df_resampled['d13c'] , mode="markers", line_color='#C20A20',text=df_sorted['Name'], name='raw'), secondary_y=False)


#fig.add_trace(go.Scatter(x=climate['age_absolute'], y= -climate['synthetic_d18o'] , mode="lines", line_color='#F54927', name='synthetic d18o', line_width=1), secondary_y=False)


#fig.add_trace(go.Scatter(x=climate['age_absolute'], y= -climate['synthetic_d18o'] , mode="lines", line_color='#F54927', name='synthetic d18o', line_width=1), secondary_y=True)
#fig.add_trace(go.Scatter(x=tide['Date'], y= tide['Sea Level'] , mode="lines", line_color='#27BBF5', name='Sea Level', line_width=2), secondary_y=True)



fig.update_layout(
    yaxis2=dict(title="Secondary Y-axis Title", overlaying="y", side="right", gridcolor='lightgray', showgrid=False)) # Grid for secondary Y-axis)
#fig.update_layout(
#    xaxis2=dict(title="Secondary X-axis Title", overlaying="x", side="top", gridcolor='lightgray', showgrid=False)) # Grid for secondary Y-axis)

#fig.update_layout(width=1700, height=800, title_text="d18O observed and d18O synthetic (climate)", template='simple_white')
fig.update_layout(width=1000, height=500, title_text="d18O observed and d18O synthetic (climate)", template='simple_white')

# Set x-axis title
fig.update_xaxes(title_text="Year")

# Set y-axes titles
fig.update_yaxes(title_text= r'$\delta^{18}$ O Observerd', secondary_y=False)
fig.update_yaxes(title_text= r'$\delta^{18}$ O Synthetic', secondary_y=True)

fig.show()
#fig.write_html('d18o_syntetic-d18O.html')


In [ ]:

fig = make_subplots(specs=[[{"secondary_y": True}]])


fig.add_trace(go.Scatter(x=df_sorted['age_final'], y=df_sorted['d13c_no_tectonic'] , mode="lines", line_color='#27F535',text=df_sorted['Name'], name='raw'), secondary_y=False)
#fig.add_trace(go.Scatter(x=df_sorted['age_final'], y=df_sorted['d13c'] , mode="lines", line_color='#27F535',text=df_sorted['Name'], name='raw'), secondary_y=False)

#fig.add_trace(go.Scatter(x=df_sorted['age_ws'], y=df_sorted['d13c_ma'] , mode="lines", line_color='#05700C', name='3 month average'), secondary_y=False)
#fig.add_trace(go.Scatter(x=df_sorted['age_ws2'], y=df_sorted['d13c_ma2'] , mode="lines", line_color='#011803', name='3 year average'), secondary_y=False)

fig.add_trace(go.Scatter(x=df_resampled['age_resampled'], y=df_resampled['d13c_no_tectonic'] , mode="markers", line_color='#C20A20',text=df_sorted['Name'], name='raw'), secondary_y=False)


fig.add_trace(go.Scatter(x=climate['age_absolute'], y= -climate['synthetic_d18o'] , mode="lines", line_color='#F54927', name='synthetic d18o', line_width=1), secondary_y=False)


#fig.add_trace(go.Scatter(x=climate['age_absolute'], y= -climate['synthetic_d18o'] , mode="lines", line_color='#F54927', name='synthetic d18o', line_width=1), secondary_y=True)
#fig.add_trace(go.Scatter(x=tide['Date'], y= tide['Sea Level'] , mode="lines", line_color='#27BBF5', name='Sea Level', line_width=2), secondary_y=True)



fig.update_layout(
    yaxis2=dict(title="Secondary Y-axis Title", overlaying="y", side="right", gridcolor='lightgray', showgrid=False)) # Grid for secondary Y-axis)
#fig.update_layout(
#    xaxis2=dict(title="Secondary X-axis Title", overlaying="x", side="top", gridcolor='lightgray', showgrid=False)) # Grid for secondary Y-axis)

#fig.update_layout(width=1700, height=800, title_text="d18O observed and d18O synthetic (climate)", template='simple_white')
fig.update_layout(width=1000, height=500, title_text="d18O observed and d18O synthetic (climate)", template='simple_white')

# Set x-axis title
fig.update_xaxes(title_text="Year")

# Set y-axes titles
fig.update_yaxes(title_text= r'$\delta^{18}$ O Observerd', secondary_y=False)
fig.update_yaxes(title_text= r'$\delta^{18}$ O Synthetic', secondary_y=True)

fig.show()
#fig.write_html('d18o_syntetic-d18O.html')


In [ ]:


fig = make_subplots(specs=[[{"secondary_y": True}]])


fig.add_trace(go.Scatter(x=df_sorted['age_final'], y=df_sorted['d18o'] , mode="markers", line_color='#3492eb',text=df_sorted['Name'], name='raw'), secondary_y=False)
fig.add_trace(go.Scatter(x=df_sorted['age_final'], y=df_sorted['d18o'] , mode="lines", line_color='#3492eb',text=df_sorted['Name'], name='raw'), secondary_y=False)

fig.add_trace(go.Scatter(x=df_sorted['age_ws'], y=df_sorted['d18o_ma'] , mode="lines", line_color='#134b80', name='3 month average'), secondary_y=False)
fig.add_trace(go.Scatter(x=df_sorted['age_ws2'], y=df_sorted['d18o_ma2'] , mode="lines", line_color='#0D0454', name='3 year average'), secondary_y=False)

fig.add_trace(go.Scatter(x=df_resampled['age_resampled'], y=df_resampled['d18o'] , mode="markers", line_color='#C20A20',text=df_sorted['Name'], name='raw'), secondary_y=False)


#fig.add_trace(go.Scatter(x=climate['age_absolute'], y= -climate['synthetic_d18o'] , mode="lines", line_color='#F54927', name='synthetic d18o', line_width=1), secondary_y=False)


#fig.add_trace(go.Scatter(x=climate['age_absolute'], y= -climate['synthetic_d18o'] , mode="lines", line_color='#F54927', name='synthetic d18o', line_width=1), secondary_y=True)
#fig.add_trace(go.Scatter(x=tide['Date'], y= tide['Sea Level'] , mode="lines", line_color='#27BBF5', name='Sea Level', line_width=2), secondary_y=True)



fig.update_layout(
    yaxis2=dict(title="Secondary Y-axis Title", overlaying="y", side="right", gridcolor='lightgray', showgrid=False)) # Grid for secondary Y-axis)
#fig.update_layout(
#    xaxis2=dict(title="Secondary X-axis Title", overlaying="x", side="top", gridcolor='lightgray', showgrid=False)) # Grid for secondary Y-axis)

fig.update_layout(width=1700, height=800, title_text="d18O observed and d18O synthetic (climate)", template='simple_white')
# Set x-axis title
fig.update_xaxes(title_text="Year")

# Set y-axes titles
fig.update_yaxes(title_text= r'$\delta^{18}$ O Observerd', secondary_y=False)
fig.update_yaxes(title_text= r'$\delta^{18}$ O Synthetic', secondary_y=True)

fig.show()
#fig.write_html('d18o_syntetic-d18O.html')


In [ ]:
df_resampled

In [ ]:
# Ensure df_resampled is sorted by date
df = df_resampled[['d13c', 'd13c_no_tectonic','d18o','d18o_no_tectonic']].copy()
df = df.sort_index()

# Target dates to align with
target_dates = pd.to_datetime(enso.loc[:685, 'Date'])

# Interpolate to get values at those dates
aligned = df.reindex(df.index.union(target_dates)).interpolate(method='time').loc[target_dates]
aligned['age_absolute'] = np.array([datetime_to_decimal_year(x) for x in pd.to_datetime(aligned.index)])
aligned['ENSO'] = enso.loc[:685, 'ENSO'].values
aligned_tidegauge = tide.reindex(tide.index.union(target_dates)).interpolate(method='time').loc[target_dates]
aligned = aligned.join(aligned_tidegauge['Sea Level'].dropna())
aligned_climate = climate.set_index('datetime')
aligned_climate = aligned_climate.reindex(aligned_climate.index.union(target_dates)).interpolate(method='time').loc[target_dates]
aligned = aligned.join(aligned_climate[['SST','SSS', 'synthetic_d18o']].dropna())

aligned.loc['2005-01-01':'2008-01-01']

In [ ]:
np.corrcoef(np.arange(10), np.arange(10)+5000)

In [ ]:
correlation_matrix = aligned.corr()

# Create a heatmap
fig = plt.figure(figsize=(6, 4))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, fmt=".2f")
plt.title("Correlation Matrix Heatmap")
plt.tight_layout()
plt.show()
#fig.savefig('corr_matrix.png')

In [ ]:
rolling_corr = aligned['d13c'].rolling(window=30, center=True).corr(aligned['synthetic_d18o'])
rolling_corr_age = [ datetime_to_decimal_year(rolling_corr.index[i]) for i in range(len(rolling_corr))]
#rolling_corr_age

In [ ]:
import matplotlib as mpl

fig, axes = plt.subplots(2,1,figsize=(20,7), sharex=True, gridspec_kw={'hspace':0})

ax = axes[0]
#ax.plot(aligned['age_absolute'], aligned['d13c_no_tectonic'], color='#27F535')
ax.plot(aligned['age_absolute'], aligned['d13c'], color='#27F535')


ax2= ax.twiny()
ax.plot(aligned['age_absolute'], aligned['synthetic_d18o'], color='blue')


ax = axes[1]

x = rolling_corr_age
y = rolling_corr.values

# --- plot correlation line ---
ax.plot(x, y, color='black', linewidth=0.8, zorder=3)

# --- create diverging colormap ---
cmap = plt.cm.RdBu_r     # blue = positive, red = negative
norm = mpl.colors.Normalize(vmin=-1, vmax=1)

# --- colored background stripes ---
for i in range(len(x)-1):
    ax.axvspan(x[i], x[i+1],
               color=cmap(norm(y[i])),
               alpha=0.6,
               linewidth=0)

ax.set_ylim(-1, 1)
ax.grid(zorder=0)

In [ ]:
rolling_corr = aligned['d13c'].rolling(window=30, center=True).corr(aligned['Sea Level'])
rolling_corr_age = [ datetime_to_decimal_year(rolling_corr.index[i]) for i in range(len(rolling_corr))]
#rolling_corr_age

In [ ]:
fig, axes = plt.subplots(2,1,figsize=(20,7), sharex=True, gridspec_kw={'hspace':0})

ax = axes[0]
#ax.plot(aligned['age_absolute'], aligned['d13c_no_tectonic'], color='#27F535')
ax.plot(aligned['age_absolute'], aligned['d13c'], color='#27F535')


ax2 = ax.twinx()
ax2.plot(aligned['age_absolute'], aligned['Sea Level'], color='cyan')


ax = axes[1]

x = rolling_corr_age
y = rolling_corr.values

# --- plot correlation line ---
ax.plot(x, y, color='black', linewidth=0.8, zorder=3)

# --- create diverging colormap ---
cmap = plt.cm.RdBu_r     # blue = positive, red = negative
norm = mpl.colors.Normalize(vmin=-1, vmax=1)

# --- colored background stripes ---
for i in range(len(x)-1):
    ax.axvspan(x[i], x[i+1],
               color=cmap(norm(y[i])),
               alpha=0.6,
               linewidth=0)

ax.set_ylim(-1, 1)
ax.grid(zorder=0)

In [ ]:
rolling_corr = aligned['d13c'].rolling(window=30, center=True).corr(aligned['ENSO'])
rolling_corr_age = [ datetime_to_decimal_year(rolling_corr.index[i]) for i in range(len(rolling_corr))]
#rolling_corr_age

In [ ]:
fig, axes = plt.subplots(2,1,figsize=(20,7), sharex=True, gridspec_kw={'hspace':0})

ax = axes[0]
#ax.plot(aligned['age_absolute'], aligned['d13c_no_tectonic'], color='#27F535')
ax.plot(aligned['age_absolute'], aligned['d13c'], color='#27F535')


ax2 = ax.twinx()
ax2.plot(aligned['age_absolute'], aligned['ENSO'], color='k')


ax = axes[1]

x = rolling_corr_age
y = rolling_corr.values

# --- plot correlation line ---
ax.plot(x, y, color='black', linewidth=0.8, zorder=3)

# --- create diverging colormap ---
cmap = plt.cm.RdBu_r     # blue = positive, red = negative
norm = mpl.colors.Normalize(vmin=-1, vmax=1)

# --- colored background stripes ---
for i in range(len(x)-1):
    ax.axvspan(x[i], x[i+1],
               color=cmap(norm(y[i])),
               alpha=0.6,
               linewidth=0)

ax.set_ylim(-1, 1)
ax.grid(zorder=0)

In [ ]:
corr_win_size = 36


fig, axes = plt.subplots( 6,1 , figsize=(15,12), sharex=True, gridspec_kw={'hspace':0})

rolling_corr_1 = aligned['d13c_no_tectonic'].rolling(window=corr_win_size, center=True).corr(aligned['synthetic_d18o'])
rolling_corr_age_1 = [ datetime_to_decimal_year(rolling_corr_1.index[i]) for i in range(len(rolling_corr))]

rolling_corr_2 = aligned['d13c_no_tectonic'].rolling(window=corr_win_size, center=True).corr(aligned['Sea Level'])
rolling_corr_age_2 = [ datetime_to_decimal_year(rolling_corr_2.index[i]) for i in range(len(rolling_corr))]

rolling_corr_3 = aligned['d13c_no_tectonic'].rolling(window=corr_win_size, center=True).corr(aligned['ENSO'])
rolling_corr_age_3 = [ datetime_to_decimal_year(rolling_corr_3.index[i]) for i in range(len(rolling_corr))]

# --- create diverging colormap ---
cmap = plt.cm.RdBu_r     # blue = positive, red = negative
norm = mpl.colors.Normalize(vmin=-1, vmax=1)


#-----------------------------------------------------------------------------

ax = axes[0]
#ax.plot(aligned['age_absolute'], aligned['d13c_no_tectonic'], color='#27F535')
ax.plot(aligned['age_absolute'], aligned['d13c_no_tectonic'], color='#27F535')
ax.set_ylabel(r'$\delta^{13}$C Observed'+'\n'+'detrended')


ax2= ax.twinx()
ax2.plot(aligned['age_absolute'], aligned['synthetic_d18o'], color='blue')
ax2.set_ylabel(r'Climate Based $\delta^{18}$O')


ax = axes[1]

x = rolling_corr_age_1
y = rolling_corr_1.values

# --- plot correlation line ---
ax.plot(x, y, color='black', linewidth=0.8, zorder=3)

# --- colored background stripes ---
for i in range(len(x)-1):
    ax.axvspan(x[i], x[i+1],
               color=cmap(norm(y[i])),
               alpha=0.6,
               linewidth=0)

ax.set_ylim(-1, 1)
ax.grid(zorder=0)


#-----------------------------------------------------------------------------------

ax = axes[2]
#ax.plot(aligned['age_absolute'], aligned['d13c_no_tectonic'], color='#27F535')
ax.plot(aligned['age_absolute'], aligned['d13c_no_tectonic'], color='#27F535')
ax.set_ylabel(r'$\delta^{13}$C Observed'+'\n'+'detrended')


ax2 = ax.twinx()
ax2.plot(aligned['age_absolute'], aligned['Sea Level'], color='red')
ax2.set_ylabel('Sea Level [m]')

ax = axes[3]

x = rolling_corr_age_2
y = rolling_corr_2.values

# --- plot correlation line ---
ax.plot(x, y, color='black', linewidth=0.8, zorder=3)



# --- colored background stripes ---
for i in range(len(x)-1):
    ax.axvspan(x[i], x[i+1],
               color=cmap(norm(y[i])),
               alpha=0.6,
               linewidth=0)

ax.set_ylim(-1, 1)
ax.grid(zorder=0)

#-----------------------------------------------------------------------------------

ax = axes[4]
#ax.plot(aligned['age_absolute'], aligned['d13c_no_tectonic'], color='#27F535')
ax.plot(aligned['age_absolute'], aligned['d13c_no_tectonic'], color='#27F535')
ax.set_ylabel(r'$\delta^{13}$C Observed'+'\n'+'detrended')


ax2 = ax.twinx()
ax2.plot(aligned['age_absolute'], aligned['ENSO'], color='k')
ax2.set_ylabel(r'ENSO Index')


ax = axes[5]

x = rolling_corr_age_3
y = rolling_corr_3.values

# --- plot correlation line ---
ax.plot(x, y, color='black', linewidth=0.8, zorder=3)

# --- colored background stripes ---
for i in range(len(x)-1):
    ax.axvspan(x[i], x[i+1],
               color=cmap(norm(y[i])),
               alpha=0.6,
               linewidth=0)

ax.set_ylim(-1, 1)
ax.grid(zorder=0)
axes[0].set_title('Rolling Correlation Window Size: {:}'.format(corr_win_size),fontsize=20)

fig.savefig('correlation_d13c.png')

In [ ]:
corr_win_size=36

fig, axes = plt.subplots( 6,1 , figsize=(15,12), sharex=True, gridspec_kw={'hspace':0})

rolling_corr_1 = aligned['d18o'].rolling(window=corr_win_size, center=True).corr(aligned['synthetic_d18o'])
rolling_corr_age_1 = [ datetime_to_decimal_year(rolling_corr_1.index[i]) for i in range(len(rolling_corr))]

rolling_corr_2 = aligned['d18o'].rolling(window=corr_win_size, center=True).corr(aligned['Sea Level'])
rolling_corr_age_2 = [ datetime_to_decimal_year(rolling_corr_2.index[i]) for i in range(len(rolling_corr))]

rolling_corr_3 = aligned['d18o'].rolling(window=corr_win_size, center=True).corr(aligned['ENSO'])
rolling_corr_age_3 = [ datetime_to_decimal_year(rolling_corr_3.index[i]) for i in range(len(rolling_corr))]

# --- create diverging colormap ---
cmap = plt.cm.RdBu_r     # blue = positive, red = negative
norm = mpl.colors.Normalize(vmin=-1, vmax=1)


#-----------------------------------------------------------------------------

ax = axes[0]
#ax.plot(aligned['age_absolute'], aligned['d13c_no_tectonic'], color='#27F535')
ax.plot(aligned['age_absolute'], aligned['d18o'], color='#3492eb')
ax.set_ylabel(r'$\delta^{18}$O Observed')


ax2= ax.twinx()
ax2.plot(aligned['age_absolute'], aligned['synthetic_d18o'], color='blue')
ax2.set_ylabel(r'Climate Based $\delta^{18}$O')


ax = axes[1]

x = rolling_corr_age_1
y = rolling_corr_1.values

# --- plot correlation line ---
ax.plot(x, y, color='black', linewidth=0.8, zorder=3)

# --- colored background stripes ---
for i in range(len(x)-1):
    ax.axvspan(x[i], x[i+1],
               color=cmap(norm(y[i])),
               alpha=0.6,
               linewidth=0)

ax.set_ylim(-1, 1)
ax.grid(zorder=0)


#-----------------------------------------------------------------------------------

ax = axes[2]
#ax.plot(aligned['age_absolute'], aligned['d13c_no_tectonic'], color='#27F535')
ax.plot(aligned['age_absolute'], aligned['d18o'], color='#3492eb')
ax.set_ylabel(r'$\delta^{18}$O Observed')


ax2 = ax.twinx()
ax2.plot(aligned['age_absolute'], aligned['Sea Level'], color='red')
ax2.set_ylabel('Sea Level [m]')

ax = axes[3]

x = rolling_corr_age_2
y = rolling_corr_2.values

# --- plot correlation line ---
ax.plot(x, y, color='black', linewidth=0.8, zorder=3)



# --- colored background stripes ---
for i in range(len(x)-1):
    ax.axvspan(x[i], x[i+1],
               color=cmap(norm(y[i])),
               alpha=0.6,
               linewidth=0)

ax.set_ylim(-1, 1)
ax.grid(zorder=0)

#-----------------------------------------------------------------------------------

ax = axes[4]
#ax.plot(aligned['age_absolute'], aligned['d13c_no_tectonic'], color='#27F535')
ax.plot(aligned['age_absolute'], aligned['d18o'], color='#3492eb')
ax.set_ylabel(r'$\delta^{18}$O Observed')


ax2 = ax.twinx()
ax2.plot(aligned['age_absolute'], aligned['ENSO'], color='k')
ax2.set_ylabel(r'ENSO Index')


ax = axes[5]

x = rolling_corr_age_3
y = rolling_corr_3.values

# --- plot correlation line ---
ax.plot(x, y, color='black', linewidth=0.8, zorder=3)

# --- colored background stripes ---
for i in range(len(x)-1):
    ax.axvspan(x[i], x[i+1],
               color=cmap(norm(y[i])),
               alpha=0.6,
               linewidth=0)

ax.set_ylim(-1, 1)
ax.grid(zorder=0)

#fig.suptitle('Rolling Correlation Window Size: {:}'.format(corr_win_size))
axes[0].set_title('Rolling Correlation Window Size: {:}'.format(corr_win_size),fontsize=20)

fig.savefig('correlation_d18o.png')

In [ ]:
rolling_corr_1 = aligned['d13c'].rolling(window=30, center=True).corr(aligned['synthetic_d18o'])
rolling_corr_2 = aligned['d13c'].rolling(window=30, center=True).corr(aligned['Sea Level'])
rolling_corr_3 = aligned['d13c'].rolling(window=30, center=True).corr(aligned['ENSO'])


In [ ]:

fig = make_subplots(specs=[[{"secondary_y": True}]])


fig.add_trace(go.Scatter(x=df_sorted['age_final'], y=df_sorted['d13c'] , mode="markers", line_color='#27F535',text=df_sorted['Name'], name='raw'), secondary_y=False)
fig.add_trace(go.Scatter(x=df_sorted['age_ws'], y=df_sorted['d13c_ma'] , mode="lines", line_color='#05700C', name='3 month average'), secondary_y=False)
fig.add_trace(go.Scatter(x=df_sorted['age_ws2'], y=df_sorted['d13c_ma2'] , mode="lines", line_color='#011803', name='3 year average'), secondary_y=False)

#fig.add_trace(go.Scatter(x=climate['age_absolute'], y= -climate['synthetic_d18o'] , mode="lines", line_color='#F54927', name='synthetic d18o', line_width=1), secondary_y=False)


#fig.add_trace(go.Scatter(x=climate['age_absolute'], y= -climate['synthetic_d18o'] , mode="lines", line_color='#F54927', name='synthetic d18o', line_width=1), secondary_y=True)
#fig.add_trace(go.Scatter(x=tide['Date'], y= tide['Sea Level'] , mode="lines", line_color='#27BBF5', name='Sea Level', line_width=2), secondary_y=True)
#fig.add_trace(go.Scatter(x=tide['Date'], y= tide['Sea Level'] , mode="lines", line_color='#27BBF5', name='Sea Level', line_width=2), secondary_y=True)
fig.add_trace(go.Scatter(x=morpho_memo['age_absolute'], y=-morpho_memo['y_true'], mode="lines", line=dict(color='purple', width=4), name='SPLA morpho memo'), secondary_y=True)


fig.add_trace(go.Scatter(x=morpho_data['date']-0.5, y=-morpho_data['SPLA_luc'], mode="lines", line=dict(color='firebrick', width=4), name='SPLA morpho luc'), secondary_y=True)
fig.add_trace(go.Scatter(x=morpho_data['date']-0.5, y=morpho_data['SPLA_fred'], mode="lines", line=dict(color='orange', width=4), name='SPLA morpho fred'), secondary_y=True)

fig.add_trace(go.Scatter(x=morpho_data['date']-0.5, y=-morpho_data['SPA_luc'], mode="lines", line=dict(color='navy', width=4), name='SPA morpho luc'), secondary_y=True)
fig.add_trace(go.Scatter(x=morpho_data['date']-0.5, y=-morpho_data['SPA_fred'], mode="lines", line=dict(color='black', width=4), name='SPA morpho fred'), secondary_y=True)




fig.update_layout(
    yaxis2=dict(title="Secondary Y-axis Title", overlaying="y", side="right", gridcolor='lightgray', showgrid=False)) # Grid for secondary Y-axis)
#fig.update_layout(
#    xaxis2=dict(title="Secondary X-axis Title", overlaying="x", side="top", gridcolor='lightgray', showgrid=False)) # Grid for secondary Y-axis)

fig.update_layout(width=1600, height=800, title_text="d18O observed and d18O synthetic (climate)", template='simple_white')
# Set x-axis title
fig.update_xaxes(title_text="Year")

# Set y-axes titles
fig.update_yaxes(title_text= r'$\delta^{13}$ O Observerd', secondary_y=False)
fig.update_yaxes(title_text= 'Sea Level [mm]', secondary_y=True)

fig.show()
#fig.write_html('d13c_all-morphologies.html')


In [ ]:
fred_resampled

In [ ]:
 fred.loc[90:,['calendar_year','d13c','d18o','d18o_pdb']].rolling(window_size1, closed='left').mean().shift(-(window_size1 ))

In [ ]:
# assign
fred.loc[:89, ['age_ma2','d13c_ma2','d18o_ma2','d18o_pdb_ma2']] = right_roll.loc[:89].to_numpy()
fred.loc[90:, ['age_ma2','d13c_ma2','d18o_ma2','d18o_pdb_ma2']] = left_roll.loc[90:].to_numpy()

In [ ]:
#fred[fred['decimal_year']<2007.25]
x = pd.concat([fred.loc[:89,['d13c','d18o','calendar_year','d18o_pdb']].rolling(window_size1, center=True).mean(),
         fred.loc[90:,['d13c','d18o','calendar_year','d18o_pdb']].rolling(window_size1).mean().shift(-(window_size1//2 ))])


x.head(20)

In [ ]:
right_roll.loc[:89].to_numpy()

In [ ]:
window_size1=12

fred = pd.read_excel('20260309_freds-data/fred_age_model_v2.xlsx').dropna()
fred['d18o_pdb'] = -(fred['d18o']-20.91)/1.03091
fred[['age_ma','d13c_ma','d18o_ma', 'd18o_pdb_ma']] = fred[['calendar_year','d13c','d18o','d18o_pdb']].rolling(window_size, center=True).mean()


cols = ['calendar_year','d13c','d18o','d18o_pdb']
new_cols = ['age_ma2','d13c_ma2','d18o_ma2','d18o_pdb_ma2']
rename_dict = dict(zip(cols,new_cols))
right_roll = fred[cols].rolling(window_size1).mean().rename(columns=rename_dict)
left_roll = fred[cols].rolling(window_size1,center=True).mean().shift(window_size1).rename(columns=rename_dict)
fred.loc[:89, new_cols] = right_roll.loc[:89].to_numpy()
fred.loc[90:, new_cols] = left_roll.loc[90:].to_numpy()

fred[['d13c_std','d18o_std']] = fred[['d13c','d18o_pdb']].rolling(10, center=True).std()
fred_resampled = fred.drop(fred.columns[4:], axis=1)
fred_resampled['age_dt'] = decimal_year_to_datetime(fred_resampled['calendar_year'])
fred_resampled.set_index('age_dt', inplace=True)
fred_resampled = fred_resampled.resample('30D').mean()
fred_resampled.interpolate(inplace=True)
fred

In [ ]:
window_size1 = 16
split = 2007.25

# split masks
before = fred["calendar_year"] <= split
after = fred["calendar_year"] > split

# rolling means
right_roll = fred["d13c"].rolling(window_size1).mean()
left_roll = fred["d13c"].rolling(window_size1).mean().shift(-(window_size1-1))

ma = pd.Series(index=fred.index, dtype=float)
ma.loc[before] = right_roll.loc[before]
ma.loc[after] = left_roll.loc[after]

In [ ]:

ma

In [ ]:

fig = make_subplots(specs=[[{"secondary_y": True}]])


fig.add_trace(go.Scatter(x=fred['calendar_year'], y=fred['d13c'] , mode="markers", line_color='#27F535', name='fred d13c'), secondary_y=False)
fig.add_trace(go.Scatter(x=fred['age_ma'], y=fred['d13c_ma'] , mode="lines", line_color='#05700C', name='moving average'), secondary_y=False)
fig.add_trace(go.Scatter(x=fred['calendar_year'], y=ma , mode="lines", line_color='#011803', name='moving average 2'), secondary_y=False)


fig.add_trace(go.Scatter(x=fred['calendar_year'], y=fred['d18o'] , mode="markers", line_color='#3492eb', name='fred d18O'), secondary_y=True)
fig.add_trace(go.Scatter(x=fred['age_ma'], y=fred['d18o_ma'] , mode="lines", line_color='#134b80', name='moving average'), secondary_y=True)

fig.update_layout(
    yaxis2=dict(title="Secondary Y-axis Title", overlaying="y", side="right", gridcolor='lightgray', showgrid=False)) # Grid for secondary Y-axis)

fig.update_layout(width=1600, height=800, title_text="d18O observed and d18O synthetic (climate)", template='simple_white')
# Set x-axis title
fig.update_xaxes(title_text="Year")

# Set y-axes titles
fig.update_yaxes(title_text= r'$\delta^{13}$ C Observerd', secondary_y=False)
fig.update_yaxes(title_text= r'$\delta^{18}$ O Observerd', secondary_y=True)


fig.show()
#fig.write_html('d13c_all-morphologies.html')


In [ ]:
window_size1=12

fred_old = pd.read_excel('../../2025-08-22_FredsData-post2008/FredData_editedMemo_raw.xlsx').dropna()
fred_old['d18o_pdb'] = -(fred['d18o']-20.91)/1.03091
fred_old[['age_ma','d13c_ma','d18o_ma', 'd18o_pdb_ma']] = fred[['calendar_year','d13c','d18o','d18o_pdb']].rolling(window_size, center=True).mean()
fred_old

In [ ]:

fig = make_subplots(specs=[[{"secondary_y": True}]])


fig.add_trace(go.Scatter(x=fred['calendar_year'], y=fred['d13c'] , mode="markers", line_color='#27F535', name='fred d13c'), secondary_y=False)
fig.add_trace(go.Scatter(x=fred['age_ma'], y=fred['d13c_ma'] , mode="lines", line_color='#05700C', name='moving average'), secondary_y=False)
#fig.add_trace(go.Scatter(x=fred['calendar_year'], y=ma , mode="lines", line_color='#011803', name='moving average 2'), secondary_y=False)
fig.add_trace(go.Scatter(x=fred_old['decimal_year'], y=fred['d13c'] , mode="markers", line_color='#F54522', name='old age model'), secondary_y=False)
fig.add_trace(go.Scatter(x=fred_old['age_ma'], y=fred['d13c_ma'] , mode="lines", line_color='#872109', name='moving average'), secondary_y=False)



#fig.add_trace(go.Scatter(x=fred['calendar_year'], y=fred['d18o'] , mode="markers", line_color='#3492eb', name='fred d18O'), secondary_y=True)
#fig.add_trace(go.Scatter(x=fred['age_ma'], y=fred['d18o_ma'] , mode="lines", line_color='#134b80', name='moving average'), secondary_y=True)

fig.update_layout(
    yaxis2=dict(title="Secondary Y-axis Title", overlaying="y", side="right", gridcolor='lightgray', showgrid=False)) # Grid for secondary Y-axis)

fig.update_layout(width=1600, height=800, title_text="d18O observed and d18O synthetic (climate)", template='simple_white')
# Set x-axis title
fig.update_xaxes(title_text="Year")

# Set y-axes titles
fig.update_yaxes(title_text= r'$\delta^{13}$ C Observerd', secondary_y=False)
fig.update_yaxes(title_text= r'$\delta^{18}$ O Observerd', secondary_y=True)


fig.show()
#fig.write_html('d13c_all-morphologies.html')
